# Notebook 01 — Environment and Data Loading

**Project:** Ten-Year Haemoglobin Genotype Surveillance in a Nigerian University Cohort
(Bowen University, 2015–2024, n = 8,890)

**Purpose of this notebook**

This is the entry point of the analysis. It sets up a reproducible environment, documents
how the raw screening exports were turned into the cleaned analytical dataset, loads that
dataset, and validates it before any analysis begins. Notebooks 02 to 06 build on the
dataset loaded here.

**What this notebook produces**

- A fixed, reproducible environment (pinned display options and random seed)
- The loaded analytical dataset (`processed_genotype_data.csv`, 8,890 rows)
- A set of validation checks that confirm the dataset matches the published cohort before
  any downstream work proceeds

**Data provenance, in brief.** Ten annual screening exports (2015–2024), each in a
different spreadsheet layout, were standardised into a single analytical file with
consistent identifier, sex, genotype, and year fields, plus a derived clinical category.
The cleaned file is the authoritative input to every notebook in this repository. The raw
exports are retained in `data/raw/` for transparency.

## Cell 1 — Environment setup

In [1]:
# Cell 1: Environment setup
# Minimal, focused imports. Heavier libraries (scikit-learn, scipy) are imported in the
# notebooks where they are actually used, keeping each notebook's dependencies explicit.

from pathlib import Path
import pandas as pd
import numpy as np

# Reproducibility: a single fixed seed used everywhere a random process appears.
RANDOM_STATE = 42

# Readable, stable DataFrame display.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:.2f}")

# Resolve project paths relative to the notebook location so the code runs unchanged on
# any machine (local laptop, a collaborator's computer, or a CI / compute capsule).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "processed_genotype_data.csv"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

print("Environment configured.")
print(f"Project:           {PROJECT_ROOT.name}")
print(f"Processed dataset: {DATA_PROCESSED.relative_to(PROJECT_ROOT)}")
print(f"Random seed:       {RANDOM_STATE}")

Environment configured.
Project:           haemoglobin-genotype-surveillance
Processed dataset: data/processed/processed_genotype_data.csv
Random seed:       42


## Cell 2 — Loading function and data provenance

The analytical dataset is loaded through a single function with a docstring that records
how the raw exports map onto the cleaned fields. Keeping the loader in one place means
every downstream notebook reads the data the same way.

The raw exports differed substantially year to year: header rows appeared on different
lines, identifier prefixes varied by intake (for example `I-` in 2020, `J-` in 2021), age
was recorded only from 2020 onward, and one 2022 record carried an out-of-range age value.
Standardising these into consistent `ID`, `Sex`, `Genotype`, and `Year` fields, and
deriving a `Clinical_Category`, produced the file loaded below.

In [2]:
# Cell 2: Loading function

VALID_GENOTYPES = ["AA", "AS", "AC", "SS", "SC", "CC"]
CLINICAL_MAP = {
    "AA": "Normal",
    "AS": "Carrier", "AC": "Carrier",
    "SS": "Disease", "SC": "Disease", "CC": "Disease",
}


def load_genotype_data(path=DATA_PROCESSED):
    """Load the cleaned analytical dataset.

    The dataset holds one row per screened student with the following fields:
        ID                 de-identified identifier (no personal information)
        Sex                F or M
        Genotype           one of AA, AS, AC, SS, SC, CC
        Year               screening year, 2015-2024
        Clinical_Category  Normal (AA); Carrier (AS, AC); Disease (SS, SC, CC)

    Returns
    -------
    pandas.DataFrame
    """
    df = pd.read_csv(path, dtype={"ID": str, "Sex": str, "Genotype": str,
                                  "Clinical_Category": str})
    df["Year"] = df["Year"].astype(int)
    return df


def add_clinical_categories(df):
    """Derive (or re-derive) the clinical category from genotype.

    Provided for transparency: it lets a reader confirm the Clinical_Category field in the
    file is exactly the documented mapping rather than something applied ad hoc.
    """
    out = df.copy()
    out["Clinical_Category"] = out["Genotype"].map(CLINICAL_MAP)
    return out


print("Loading function defined.")

Loading function defined.


## Cell 3 — Load and inspect the dataset

In [3]:
# Cell 3: Load the analytical dataset and inspect its basic structure

df = load_genotype_data()

print(f"Rows: {len(df):,}    Columns: {df.shape[1]}")
print()
print("Column types:")
print(df.dtypes.to_string())
print()
print("Missing values per column:")
print(df.isna().sum().to_string())
print()
print("First five records:")
display(df.head())

Rows: 8,890    Columns: 5

Column types:
ID                   object
Sex                  object
Genotype             object
Year                  int64
Clinical_Category    object

Missing values per column:
ID                   13
Sex                   0
Genotype              0
Year                  0
Clinical_Category     0

First five records:


,ID,Sex,Genotype,Year,Clinical_Category
0,D-0003,F,AA,2015,Normal
1,D-0001,F,AA,2015,Normal
2,D-0002,F,AA,2015,Normal
3,D-0004,M,AA,2015,Normal
4,D-0007,F,AA,2015,Normal


## Cell 4 — Validation

Before analysis begins, the dataset is validated against predefined cohort properties. If any check fails, execution stops to prevent invalid data from propagating through subsequent notebooks. These checks ensure that all downstream analyses (notebooks 02–06) are performed on the same 8,890-record cohort described in the manuscript.


In [6]:
# Cell 4: Validation checks

checks = []

# 1. Cohort size
checks.append(("Total records == 8,890", len(df) == 8890))

# 2. Year coverage
expected_years = set(range(2015, 2025))
checks.append(("Years span 2015-2024", set(df["Year"].unique()) == expected_years))

# 3. Genotype vocabulary
checks.append(("Only valid genotypes present",
               set(df["Genotype"].unique()).issubset(set(VALID_GENOTYPES))))

# 4. Sex vocabulary
checks.append(("Sex is F or M only", set(df["Sex"].unique()).issubset({"F", "M"})))

# 5. Clinical category matches the documented mapping exactly
rederived = add_clinical_categories(df)
checks.append(("Clinical_Category matches genotype mapping",
               (rederived["Clinical_Category"] == df["Clinical_Category"]).all()))

# 6. No missing values in the analytical fields (Sex, Genotype, Year, Clinical_Category).
#    ID is excluded here: it is a de-identified label, not an analytical variable.
analytical_fields = ["Sex", "Genotype", "Year", "Clinical_Category"]
checks.append(("No missing values in analytical fields",
               df[analytical_fields].isna().sum().sum() == 0))

results = pd.DataFrame(checks, columns=["Check", "Passed"])
display(results)

# Informational note (not a failure): a small number of records lack an ID label while
# carrying complete analytical data. They are retained because every analysis in this
# project uses Sex, Genotype, Year, and Clinical_Category, none of which are missing.
n_missing_id = df["ID"].isna().sum()
print(f"\nNote: {n_missing_id} records have a missing ID label "
      f"({n_missing_id / len(df) * 100:.2f}% of the cohort); "
      f"all analytical fields for these records are present.")

assert results["Passed"].all(), "Validation failed; stopping before analysis."
print("All validation checks passed. Dataset is ready for analysis.")

,Check,Passed
0,"Total records == 8,890",True
1,Years span 2015-2024,True
2,Only valid genotypes present,True
3,Sex is F or M only,True
4,Clinical_Category matches genotype mapping,True
5,No missing values in analytical fields,True



Note: 13 records have a missing ID label (0.15% of the cohort); all analytical fields for these records are present.
All validation checks passed. Dataset is ready for analysis.


# Cell 5: Reference distribution table

This cell generates the primary genotype distribution table used in the manuscript, including the headline percentage breakdown. It serves as a stable reference output for downstream analysis and reporting.


In [8]:

genotype_summary = (
    df["Genotype"].value_counts()
    .reindex(VALID_GENOTYPES)
    .rename("Count")
    .to_frame()
)
genotype_summary["Percent"] = (genotype_summary["Count"] / len(df) * 100).round(2)
display(genotype_summary)

genotype_summary.to_csv(OUTPUTS_TABLES / "01_genotype_distribution.csv")
print(f"Saved: outputs/tables/01_genotype_distribution.csv")
print(f"HbAA: {genotype_summary.loc['AA', 'Percent']}%  (manuscript: 75.11%)")
print(f"SCD (HbSS + HbSC): "
      f"{(genotype_summary.loc['SS','Percent'] + genotype_summary.loc['SC','Percent']):.2f}%  "
      f"(manuscript: 1.18%)")

,Count,Percent
Genotype,,
AA,6677,75.11
AS,1814,20.40
AC,292,3.28
SS,63,0.71
SC,42,0.47
CC,2,0.02


Saved: outputs/tables/01_genotype_distribution.csv
HbAA: 75.11%  (manuscript: 75.11%)
SCD (HbSS + HbSC): 1.18%  (manuscript: 1.18%)


## Summary

The environment is initialized with a fixed random seed. The analytical dataset is then loaded and checked against predefined cohort properties. The reference genotype distribution is computed and saved as the first output table.

**Next:** Notebook 02 performs descriptive exploration, including sex and year cross-tabulations, clinical category breakdowns, and age summaries for records where age is available.
